In [ ]:
# ── large-scale fork ─────────────────────────────────────────────────────────
# A copy of LEAN_LLM_OPT_4.1_Air_NRM.ipynb, edited for the large_scale dataset
# (90 ODs, 268 departures, 4 fare families, per-flight capacity). The small-scale
# notebook at the repository root is untouched; the two are deliberately NOT
# merged for now.
#
# Every path below is written relative to the repository root, so move there
# first. This notebook lives three levels down, hence the three parents.
import os, sys
from pathlib import Path
_ROOT = Path.cwd()
while not (_ROOT / "exp_config.yaml").exists() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
if not (_ROOT / "exp_config.yaml").exists():
    raise RuntimeError("run this notebook from inside the lean-llm-opt-main tree")
os.chdir(_ROOT)
sys.path.insert(0, str(_ROOT))
print("working directory:", os.getcwd())


In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.vectorstores import FAISS
import pandas as pd
from langchain_classic.chains import RetrievalQA
from langchain_classic.agents import initialize_agent, AgentType, Tool
from langchain_core.documents import Document
import openai
from datetime import datetime
import re
from datetime import time as time_class
# ---------------------------------------------------------------------------
# Unified experiment configuration + run logging  (see PATCH_GUIDE.md)
# ---------------------------------------------------------------------------
import os
import leanopt_exp as lx

# API keys come from a .env file at the repository root (see .env.example),
# from the shell environment, or -- if neither is set -- from a hidden prompt.
# Nothing is hard-coded here and nothing is written to the run log.
try:
    from dotenv import load_dotenv
    # override=True: the .env file wins over anything already in the
    # environment. Without it, a stale key left in os.environ by an earlier
    # run silently shadows the value you just edited into .env.
    load_dotenv(override=True)
except ImportError:
    pass

CFG = lx.load_config(
    "exp_config.yaml",
    model_profile='gpt-4.1',
    method='LEAN-LLM-OPT',
    dataset='Air-NRM-CA-LS',
    # Repeats. exp_config sets n_repeats: 3 but nothing was reading it here, so
    # every run landed in the same __r0__ bucket and overwrote the last. Taking
    # it from the environment keeps the default at 0 -- no existing invocation
    # changes -- while
    #     for r in 0 1 2; do LEANOPT_REPEAT=$r python run_notebook.py ...; done
    # produces three separately logged runs that aggregate_runs.py can pool into
    # mean +/- std. Worth doing here: temperature is 0.0 and the seed is fixed,
    # yet instance 13 was wrong in one run and right in the next, so a single
    # run cannot tell a stable failure from a coin flip.
    repeat_index=int(os.environ.get("LEANOPT_REPEAT", "0")),
)
lx.ensure_api_keys(CFG)                          # only for providers this profile uses
user_api_key = os.environ.get("OPENAI_API_KEY", "")   # legacy references

LOG = lx.RunLogger(CFG)
EMBEDDINGS = lx.build_embeddings(CFG)

# Problem Classification

In [ ]:
def Classification_Agent(file_path="Large_Scale_Or_Files/RefData.csv"):
    # Initialize the LLM
    llm1 = lx.build_llm(CFG, "classifier")

    # Load and process the data
    # classification only needs the problem description + its class label
    refdata = lx.load_refdata_docs(CFG)
    refdocuments = refdata

    embeddings = EMBEDDINGS
    vectors1 = FAISS.from_documents(refdocuments, embeddings)
    retriever1 = lx.build_retriever(CFG, vectors1, "refdata")
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm1,
        chain_type="stuff",
        retriever=retriever1,
        return_source_documents=True,
    )

    qa_tool1 = Tool(
        name="FileQA",
        func=qa_chain.invoke,
        description=(
            "Use this tool to answer questions about the problem type of the text. "
            "Provide the question as input, and the tool will retrieve the relevant information from the file and use it to answer the question."
        ),
    )

    few_shot_examples = """

    Question: What is the problem type in operation of the text? Please give the answer directly. Text:There are three best-selling items (P1, P2, P3) on Amazon with the profit w_1,w_2,w_3.There is an independent demand stream for each of the products. The objective of the company is to decide which demands to be fufilled over a ﬁnite sales horizon [0,10] to maximize the total expected revenue from ﬁxed initial inventories. The on-hand inventories for the three items are c_1,c_2,c_3 respectively. During the sales horizon, replenishment is not allowed and there is no any in-transit inventories. Customers who want to purchase P1,P2,P3 arrive at each period accoring to a Poisson process with a_1,a_2,a_3 the arrival rates respectively. Decision variables y_1,y_2,y_3 correspond to the number of requests that the firm plans to fulfill for product 1,2,3. These variables are all positive integers.

    Thought: I need to determine the problem type of the text. I'll use the FileQA tool to retrieve the relevant information.

    Action: FileQA

    Action Input: "What is the problem type in operation of the text? text:There are three best-selling items (P1, P2, P3) on Amazon with the profit w_1, w_2, w_3. ..."

    Observation: The problem type of the text is Network Revenue Management.

    Final Answer: Network Revenue Management.

    """

    prefix = f"""You are a helpful assistant that can answer questions about operation problems.

    Use the following examples as a guide. Always use the FileQA tool when you need to retrieve information from the file:


    {few_shot_examples}

    When you need to find information from the file, use the provided tools. And answer the question by given the answer directly. For example,

    """

    suffix = """

    Begin!

    Question: {input}
    {agent_scratchpad}"""

    agent_pc = initialize_agent(
        tools=[qa_tool1],
        llm=llm1,
        agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        **lx.agent_kwargs(CFG, prefix.format(few_shot_examples=few_shot_examples), suffix),
    )

    openai.api_request_timeout = 60 
    return agent_pc

def Problemtype(query):
    agent_pc = Classification_Agent(file_path="Large_Scale_Or_Files/RefData.csv")
    category_original=agent_pc.invoke(f"What is the problem type in operation of the text? text:{query}")
    type_output = category_original['output']
    return type_output

# ProblemSolvingModel

## PreProcess: Sharing Functions

In [ ]:
# LARGE-SCALE FORK: reads Test_Dataset/Air_NRM/large_scale/ and additionally
# loads flight_capacity.csv, which the small-scale case does not have (there,
# every flight is 187 seats and the number is hard-coded).
#
# CACHE 1 of 2 -- the CSVs. LoadFiles() is called from New_Vectors_Flight,
# New_Vectors_Demand and retrieve_parameter, i.e. several times per flight per
# query. Re-reading five CSVs each time is pure waste; they never change during
# a run. Same pattern as _SCHEDULE_CACHE in build_air_nrm_inputs.py.
_LS_DIR = 'Test_Dataset/Air_NRM/large_scale'
_FILES_CACHE = None


def LoadFiles():
  """v1, v2, od_demand, flight, flight_capacity -- read once, then reused."""
  global _FILES_CACHE
  if _FILES_CACHE is None:
      _FILES_CACHE = (
          pd.read_csv(f'{_LS_DIR}/v1.csv'),
          pd.read_csv(f'{_LS_DIR}/v2.csv'),
          pd.read_csv(f'{_LS_DIR}/od_demand.csv'),
          pd.read_csv(f'{_LS_DIR}/flight.csv'),
          pd.read_csv(f'{_LS_DIR}/flight_capacity.csv'),
      )
  return _FILES_CACHE


def LoadProducts():
  """Fare families, in the order v1.csv lists them.

  The small-scale notebook hard-codes Eco_flexi and Eco_lite. Here there are
  four, and reading them off the column names means the list can never drift
  from the data file.
  """
  v1 = LoadFiles()[0]
  seen = []
  for column in v1.columns:
      if '*' in column:
          name = column.split('*')[0]
          if name not in seen:
              seen.append(name)
  return seen


In [ ]:
# CACHE 2 of 2 -- the FAISS indexes.
#
# The originals rebuilt the index on every call. Note the `query` argument was
# never used in the body: the index depends only on flight.csv / od_demand.csv,
# so every rebuild produced a byte-identical object at full price. flight.csv
# has 1072 rows here against 38 in the small-scale set, and csv_qa_tool_CA is
# called at least twice per query (few-shot example + the agent's tool call),
# so 15 queries meant 15 x 2 x 1072 = 32,160 embeddings, 31,088 of them
# redundant. Built once, it is 1,072.
#
# This does NOT reduce the number of retrievals: each lookup embeds a different
# query string. That is what the lru_cache below is for.
_FLIGHT_INDEX = None
_DEMAND_INDEX = None


def New_Vectors_Flight(query=None):
  global _FLIGHT_INDEX
  if _FLIGHT_INDEX is None:
      v1, v2, demand, df, capacity = LoadFiles()
      new_docs = []
      for _, row in df.iterrows():
          new_docs.append(Document(
              page_content=f"OD={row['Oneway_OD']},Departure_Time_Flight1={row['Departure Time']},Oneway_Product={row['Oneway_Product']},avg_price={row['Avg Price']}",
              metadata={
                  'OD': row['Oneway_OD'],
                  'time': row['Departure Time'],
                  'product': row['Oneway_Product'],
                  'avg_price': row['Avg Price']
              }
          ))
      _FLIGHT_INDEX = FAISS.from_documents(new_docs, EMBEDDINGS)
  return _FLIGHT_INDEX


def New_Vectors_Demand(query=None):
  global _DEMAND_INDEX
  if _DEMAND_INDEX is None:
      v1, v2, demand, df, capacity = LoadFiles()
      new_docs = []
      for _, row in demand.iterrows():
          new_docs.append(Document(
              page_content=f"OD={row['Oneway_OD']}, avg_pax={row['Avg Pax']}",
              metadata={
                  'OD': row['Oneway_OD'],
                  'avg_pax': row['Avg Pax']
              }
          ))
      _DEMAND_INDEX = FAISS.from_documents(new_docs, EMBEDDINGS)
  return _DEMAND_INDEX


# CACHE 2b -- the retrieval results themselves.
#
# The index cache above removes repeated index builds but not repeated lookups.
# Across the 15 queries there are 898 flight mentions but only 248 distinct
# (OD, departure time) pairs, so the same question gets asked ~3.6x on average.
# Same index + same query string = same document, so memoising is mechanical:
# it changes nothing about what the pipeline does, it just stops paying twice.
# 3,592 retrievals -> 992.
from functools import lru_cache


def _parse_doc_field(content, field):
  """Pull one `field=value` out of a retrieved page_content."""
  for pair in [p.strip().replace('"', "'") for p in re.split(r',\s*(?=\w+=)', content)]:
      key, _, value = pair.partition('=')
      if key == field:
          return value
  return None


# One retrieval per FLIGHT, not per (flight, fare family).
#
# langchain's FAISS filter is applied AFTER the nearest-neighbour search: it
# takes the fetch_k closest documents (default 20), filters those, and returns
# up to k. With the small-scale corpus of 38 documents, 20 is over half of
# everything, so the two rows for a flight were nearly always among them. With
# 1072 documents the 20 nearest to
#     "OD=..., Departure Time=..., Oneway_Product=Eco_standard, avg_price="
# may hold only the wrong fare family -- the first live run returned Eco_lite's
# 239.38 where Eco_standard's 457.46 was asked for -- or none at all, which
# came back as None and died in float().
#
# Two changes. fetch_k is set to the corpus size so the filter is exact rather
# than a filter over an arbitrary 20-document window. And k is the number of
# fare families, so one call returns all of a flight's rows and the product is
# then claimed by name instead of by embedding rank -- the four page_contents
# differ by one word, which is not something cosine distance should be asked to
# adjudicate. Retrieval still narrows; it just no longer decides.
#
# Side effect: 4x fewer retrieval calls (3,592 -> 898 before memoisation).
@lru_cache(maxsize=None)
def retrieve_flight_prices(od, time):
  """{product: avg_price} for one flight, from a single filtered retrieval."""
  products = LoadProducts()
  # k and fetch_k come from exp_config.yaml:retrievers.air_flight_ls, so they
  # reach config_hash() and the run_id. Only the filter is per-call.
  retriever = lx.build_retriever(
      CFG, New_Vectors_Flight(), "air_flight_ls",
      search_kwargs={"filter": {"OD": od, "time": time}})
  docs = retriever.invoke(f"OD={od}, Departure Time={time}, avg_price=")

  found = {}
  for doc in docs:
      name = _parse_doc_field(doc.page_content, 'Oneway_Product')
      value = _parse_doc_field(doc.page_content, 'avg_price')
      if name is not None and value is not None:
          found[name] = value

  missing = [p for p in products if p not in found]
  if missing:
      raise LookupError(
          f"retrieval for {od} {time} returned {sorted(found)} -- no row for "
          f"{missing}. The filter is applied after the nearest-neighbour "
          f"search, so check fetch_k against the corpus size.")
  return found


def retrieve_avg_price(od, time, product):
  """One fare family's avg_price. Claimed by name from the flight's rows."""
  return retrieve_flight_prices(od, time)[product]


# Same treatment as the flight lookup. The demand corpus is 90 documents that
# read "OD=('865', '678'), avg_pax=2279.18" -- near-identical strings differing
# only in six digits. The original searched them with k=1 and no filter at all,
# which asks cosine distance to tell 90 ODs apart. It has metadata, so use it:
# filter exactly on the OD, fetch_k over the whole corpus, and check the answer.
@lru_cache(maxsize=None)
def retrieve_avg_pax(od):
  """avg_pax for one OD, filtered exactly and verified against od_demand.csv."""
  v1, v2, demand, flight_df, capacity = LoadFiles()
  retriever = lx.build_retriever(
      CFG, New_Vectors_Demand(), "air_demand_ls",
      search_kwargs={"filter": {"OD": od}})
  docs = retriever.invoke(f"OD={od}, avg_pax=")
  for doc in docs:
      value = _parse_doc_field(doc.page_content, 'avg_pax')
      if value is None:
          continue
      expected = float(demand.set_index('Oneway_OD')['Avg Pax'][od])
      if abs(float(value) - expected) > 1e-9:
          raise AssertionError(
              f"retrieval returned avg_pax {value} for {od}, but od_demand.csv "
              f"says {expected} -- the search matched a different OD")
      return value
  raise LookupError(f"retrieval found no od_demand row for {od}")


def retrieve_key_information(query):
    flight_pattern = r"\(OD\s*=\s*\(\s*'(\w+)'\s*,\s*'(\w+)'\s*\)\s+AND\s+Departure\s+Time='(\d{1,2}:\d{2})'\)"
    matches = re.findall(flight_pattern, query)

    flight_strings = [f"(OD = ('{origin}', '{destination}') AND Departure Time='{departure_time}')" for origin, destination, departure_time in matches]

    new_query = "Retrieve avg_price, avg_pax, value_list, ratio_list, value_0_list, ratio_0_list, and flight_capacity for the following flights:\n" + ", ".join(flight_strings) + "."

    # Carry the capacity sentences through.
    #
    # The agent calls CName first and feeds its output to CSVQA, so whatever
    # this drops, csv_qa_tool_CA never sees. The original kept only the flight
    # list, which is why the small-scale tool had
    #     else: eco_flex_capacity = 1.2
    # -- a default that silently replaced whatever the question actually said.
    # Here the consumption coefficients differ per query and there are four of
    # them, so they have to survive the rewrite.
    tail = re.search(r"Each flight's capacity is.*", query, re.S)
    if tail:
        new_query += "\n" + tail.group().strip()

    return new_query
def retrieve_time_period(departure_time):
    intervals = {
        '12pm~6pm': (time_class(12, 0), time_class(18, 0)),
        '6pm~10pm': (time_class(18, 0), time_class(22, 0)),
        '10pm~8am': (time_class(22, 0), time_class(8, 0)),
        '8am~12pm': (time_class(8, 0), time_class(12, 0))
    }

    if isinstance(departure_time, str):
        try:
            hours, minutes = map(int, departure_time.split(':'))
            departure_time = time_class(hours, minutes)
        except ValueError:
            raise ValueError("Time format should be: 'HH:MM'")

    for interval_name, (start, end) in intervals.items():
        if start < end:
            if start <= departure_time < end:
                return interval_name
        else:
            if departure_time >= start or departure_time < end:
                return interval_name

    return "Unknown"
def retrieve_parameter(O, time_interval, product):
    """v1 cell, v2 cell, v1 no_purchase, v2 no_purchase for one (OD, window, product).

    The original returned 0 for anything it could not find. With two products
    whose names are far apart that is survivable; with four it is not -- a
    single typo or an unbucketable departure time would silently zero out a
    coefficient and the model would still solve, just wrongly. So every lookup
    that misses now raises.
    """
    v1, v2, demand, df, capacity = LoadFiles()
    if time_interval == "Unknown":
        raise ValueError(f"departure time did not fall in any window for OD={O}")
    key = product + '*' + f'({time_interval})'

    subset = v1[v1['OD Pairs'] == O]
    subset2 = v2[v2['OD Pairs'] == O]
    if subset.empty or subset2.empty:
        raise KeyError(f"OD {O!r} is not a row of v1.csv / v2.csv")
    if key not in subset.columns or key not in subset2.columns:
        raise KeyError(f"column {key!r} is in neither v1.csv nor v2.csv; "
                       f"known products are {LoadProducts()}")

    # float(), not the numpy scalar pandas hands back. These values are printed
    # into the observation string the agent reads, and under numpy 2 a numpy
    # scalar's repr is "np.float64(0.103396)" -- which would land in the prompt
    # verbatim, cost tokens, and give the model something it cannot parse.
    return (float(subset[key].values[0]),
            float(subset2[key].values[0]),
            float(subset['no_purchase'].values[0]),
            float(subset2['no_purchase'].values[0]))


def generate_coefficients(OD, time):
    """All fare families for one (OD, departure time).

    The small-scale version called retrieve_parameter twice, once for Eco_flexi
    and once for Eco_lite, and returned a flat 6-tuple. There are four families
    here, so it returns dicts keyed by product name instead -- the shape no
    longer depends on how many products the dataset happens to have.

        values  {product: v1 cell}      -> the SBLP scale constraint
        ratios  {product: v2 cell}      -> the SBLP balance constraint
        value_0                          v1 no_purchase  (the v_0 the scale
                                         constraint divides by; it is 1 in the
                                         small-scale files and is NOT here)
        ratio_0                          v2 no_purchase
    """
    departure_time = datetime.strptime(time, '%H:%M').time()
    time_interval = retrieve_time_period(departure_time)

    values, ratios = {}, {}
    value_0 = ratio_0 = None
    for product in LoadProducts():
        value, ratio, v0, r0 = retrieve_parameter(OD, time_interval, product)
        values[product] = value
        ratios[product] = ratio
        # Same for every product of an OD; assert rather than quietly overwrite.
        if value_0 is not None and (v0 != value_0 or r0 != ratio_0):
            raise AssertionError(f"no_purchase differs across products for {OD}")
        value_0, ratio_0 = v0, r0

    return values, ratios, value_0, ratio_0


def clean_text_preserve_newlines(text):
    cleaned = re.sub(r'\x1b\[[0-9;]*[mK]', '', text)
    cleaned = re.sub(r'[^\x20-\x7E\n]', '', cleaned)
    cleaned = re.sub(r'(\n\s+)(\w+\s*=)', r'\n\2', cleaned)
    cleaned = re.sub(r'\[\s+', '[', cleaned)
    cleaned = re.sub(r'\s+\]', ']', cleaned)
    cleaned = re.sub(r',\s+', ', ', cleaned)

    return cleaned


## Flow

In [ ]:
def csv_qa_tool_flow(query: str):
    new_vectors = New_Vectors_Flight(query)
    matches = re.findall(r"\(OD\s*=\s*(\(\s*'[^']+'\s*,\s*'[^']+'\s*\))\s+AND\s+Departure\s*Time\s*=\s*'(\d{1,2}:\d{2})'\)", query)
    num_match = re.search(r"optimal (\d+) flights", query, re.IGNORECASE)
    num_flights = int(num_match.group(1)) if num_match else 4

    capacity_match = re.search(r"Eco_flex ticket consumes (\d+\.?\d*)", query)
    eco_flex_capacity = float(capacity_match.group(1)) if capacity_match else 1.6

    if matches == []:
        pattern = r"\(\('(\w+)','(\w+)'\),\s*'(\d{1,2}:\d{2})'\)"
        matches_2 = re.findall(pattern, query)

    sigma_inflow_A = []
    sigma_outflow_A = []
    sigma_inflow_B = []
    sigma_outflow_B = []
    sigma_inflow_C = []
    sigma_outflow_C = []

    if matches == []:
        matches = matches_2
        for origin,destination,time in matches:
            if origin == 'A':
                flight_name = f"({origin}{destination},{time})"
                sigma_inflow_A.append(flight_name)
            elif origin == 'B':
                flight_name = f"({origin}{destination},{time})"
                sigma_inflow_B.append(flight_name)
            elif origin == 'C':
                flight_name = f"({origin}{destination},{time})"
                sigma_inflow_C.append(flight_name)
            elif destination == 'A':
                flight_name = f"({origin}{destination},{time})"
                sigma_outflow_A.append(flight_name)
            elif destination == 'B':
                flight_name = f"({origin}{destination},{time})"
                sigma_outflow_B.append(flight_name)
            elif destination == 'C':
                flight_name = f"({origin}{destination},{time})"
                sigma_outflow_C.append(flight_name)
    
    else:
        a_origin_flights_A_out = [
            (od, time)
            for (od, time) in matches
            if od[2] == 'A'
        ]

        a_origin_flights_B_out = [
            (od, time)
            for (od, time) in matches
            if od[2] == 'B'
        ]

        a_origin_flights_C_out = [
            (od, time)
            for (od, time) in matches
            if od[2] == 'C'
        ]

        a_origin_flights_A = [
            (od, time)
            for (od, time) in matches
            if od[7] == 'A'
        ]

        a_origin_flights_B = [
            (od, time)
            for (od, time) in matches
            if od[7] == 'B'
        ]

        a_origin_flights_C = [
            (od, time)
            for (od, time) in matches
            if od[7] == 'C'
        ]

        for od, time in a_origin_flights_A:
            origin = od[2]
            destination = od[7]
            flight_name = f"({origin}{destination},{time})"
            sigma_inflow_A.append(flight_name)

        for od, time in a_origin_flights_B:
            origin = od[2]
            destination = od[7]
            flight_name = f"({origin}{destination},{time})"
            sigma_inflow_B.append(flight_name)

        for od, time in a_origin_flights_C:
            origin = od[2]
            destination = od[7]
            flight_name = f"({origin}{destination},{time})"
            sigma_inflow_C.append(flight_name)

        for od, time in a_origin_flights_A_out:
            origin = od[2]
            destination = od[7]
            flight_name = f"({origin}{destination},{time})"
            sigma_outflow_A.append(flight_name)

        for od, time in a_origin_flights_B_out:
            origin = od[2]
            destination = od[7]
            flight_name = f"({origin}{destination},{time})"
            sigma_outflow_B.append(flight_name)

        for od, time in a_origin_flights_C_out:
            origin = od[2]
            destination = od[7]
            flight_name = f"({origin}{destination},{time})"
            sigma_outflow_C.append(flight_name)

    avg_price, x, x_o, ratio_0_list, ratio_list, value_list, value_0_list, avg_pax = {}, {}, {}, {}, {}, {}, {}, {}
    y = {}
    N_l = {"AB":[],"AC":[],"BA":[],"CA":[]}

    if matches == []:
        matches = matches_2
        for origin,destination,time in matches:
            od = str((origin, destination))
            code_f = f"({origin}{destination},{time},f)"
            code_l = f"({origin}{destination},{time},l)"
            code_o = f"{origin}{destination}"
            x[code_f] = f"x_{origin}{destination}_{time}_f"
            x[code_l] = f"x_{origin}{destination}_{time}_l"
            code_y = f"({origin}{destination},{time})"
            y[code_y] = f"y_{origin}{destination}_{time}"
            retriever = lx.build_retriever(CFG, new_vectors, "air_flight", search_kwargs={"filter": {"OD": od, "time": time}})

            doc_1= retriever.invoke(f"OD={od}, Departure Time={time}, Oneway_Product=Eco_flexi, avg_price=")
            for doc in doc_1:
                content = doc.page_content
                pattern = r',\s*(?=\w+=)'
                parts = re.split(pattern, content)

                pairs = [p.strip().replace('"', "'") for p in parts]
                for pair in pairs:
                    key, value = pair.split('=')
                    if key == 'avg_price':
                        avg_price[code_f] = value


            doc_2= retriever.invoke(f"OD={od}, Departure Time={time}, Oneway_Product=Eco_lite, avg_price=")
            for doc in doc_2:
                content = doc.page_content
                pattern = r',\s*(?=\w+=)'
                parts = re.split(pattern, content)

                pairs = [p.strip().replace('"', "'") for p in parts]
                for pair in pairs:
                    key, value = pair.split('=')
                    if key == 'avg_price':
                        avg_price[code_l] = value

            value_1,ratio_1,value_2,ratio_2,value_0,ratio_0 = generate_coefficients(od,time)

            ratio_list[code_f] = ratio_1
            ratio_list[code_l] = ratio_2
            value_list[code_f] = value_1
            value_list[code_l] = value_2
            ratio_0_list[code_o] = ratio_0
            value_0_list[code_o] = value_0
    else:
        for match in matches:
            origin = match[0][2]
            destination = match[0][7]
            time = match[1]
            od = str((origin, destination))
            code_f = f"({origin}{destination},{time},f)"
            code_l = f"({origin}{destination},{time},l)"
            code_o = f"{origin}{destination}"
            x[code_f] = f"x_{origin}{destination}_{time}_f"
            x[code_l] = f"x_{origin}{destination}_{time}_l"
            code_y = f"({origin}{destination},{time})"
            y[code_y] = f"y_{origin}{destination}_{time}"
            retriever = lx.build_retriever(CFG, new_vectors, "air_flight", search_kwargs={"filter": {"OD": od, "time": time}})

            doc_1= retriever.invoke(f"OD={od}, Departure Time={time}, Oneway_Product=Eco_flexi, avg_price=")
            for doc in doc_1:
                content = doc.page_content
                pattern = r',\s*(?=\w+=)'
                parts = re.split(pattern, content)

                pairs = [p.strip().replace('"', "'") for p in parts]
                for pair in pairs:
                    key, value = pair.split('=')
                    if key == 'avg_price':
                        avg_price[code_f] = value


            doc_2= retriever.invoke(f"OD={od}, Departure Time={time}, Oneway_Product=Eco_lite, avg_price=")
            for doc in doc_2:
                content = doc.page_content
                pattern = r',\s*(?=\w+=)'
                parts = re.split(pattern, content)

                pairs = [p.strip().replace('"', "'") for p in parts]
                for pair in pairs:
                    key, value = pair.split('=')
                    if key == 'avg_price':
                        avg_price[code_l] = value

            value_1,ratio_1,value_2,ratio_2,value_0,ratio_0 = generate_coefficients(od,time)

            ratio_list[code_f] = ratio_1
            ratio_list[code_l] = ratio_2
            value_list[code_f] = value_1
            value_list[code_l] = value_2
            ratio_0_list[code_o] = ratio_0
            value_0_list[code_o] = value_0


    od_matches = re.findall(
    r"OD\s*=\s*\(\s*'([^']+)'\s*,\s*'([^']+)'\s*\)", 
    query
    )
    if od_matches == []:
        pattern = r"\(\('(\w+)','(\w+)'\)"
        matches = re.findall(pattern, query)
        od_matches = matches
    
    od_matches = list(set(od_matches))

    new_vectors_demand = New_Vectors_Demand(query)
    for origin, dest in od_matches:
        od = str((origin, dest))
        code_o = f"{origin}{dest}"
        x_o[code_o] = f"x_{origin}{dest}_o"
        retriever = lx.build_retriever(CFG, new_vectors_demand, "air_demand")
    
        doc_1= retriever.invoke(f"OD={od}, avg_pax=")
        content = doc_1[0].page_content

        pattern = r',\s*(?=\w+=)'
        parts = re.split(pattern, content)
        pairs = [p.strip().replace('"', "'") for p in parts]

        for pair in pairs:
            key, value = pair.split('=')
            if key == 'avg_pax':
                avg_pax[code_o] = value
            
    doc = f"y = {y}\n"
    doc = f"avg_price={avg_price} \n value_list ={value_list}\n ratio_list={ratio_list}\n"
    doc += f"value_0_list={value_0_list}\n ratio_0_list={ratio_0_list}\n"
    doc += f"avg_pax={avg_pax}\n"
    doc += f"capacity_consum={eco_flex_capacity}\n"
    doc += f"option_num={num_flights}\n"
    doc += f"sigma_inflow_A={sigma_inflow_A}\n"
    doc += f"sigma_outflow_A={sigma_outflow_A}\n"
    doc += f"sigma_inflow_B={sigma_inflow_B}\n"
    doc += f"sigma_outflow_B={sigma_outflow_B}\n"
    doc += f"sigma_inflow_C={sigma_inflow_C}\n"
    doc += f"sigma_outflow_C={sigma_outflow_C}\n"
    doc += f"M = 10000000\n"
    doc += f"flight_capacity=187\n"

    return doc

def retrieve_similar_docs(query,retriever):
    
    similar_docs = retriever.invoke(query)

    results = []
    for doc in similar_docs:
        results.append({
            "content": doc.page_content,
            "metadata": doc.metadata
        })
    return results

def FlowAgent(query):
    loader = CSVLoader(file_path="Large_Scale_Or_Files/RAG_Example_SBLP_Flow.csv", encoding="utf-8")
    data = loader.load()
    documents = data
    embeddings = EMBEDDINGS
    vectors = FAISS.from_documents(documents, embeddings)
    retriever = lx.build_retriever(CFG, vectors, "air_examples")
    similar_results = retrieve_similar_docs(query,retriever)
    problem_description = similar_results[0]['content'].replace("prompt:", "").strip()  
    example_matches = retrieve_key_information(problem_description)
    example_data_description = csv_qa_tool_flow(example_matches)
    example_data_description = example_data_description.replace('{', '{{')
    example_data_description = example_data_description.replace('}', '}}')    
    fewshot_example = f'''
    Question: {problem_description}

    Thought: I need to retrive the required OD and Departure Time information.

    Action: CName

    Action Input: {problem_description}

    Observation: {example_matches}

    Thought: I need to extract all relevant information from the files 'flight.csv', 'od_demand.csv', 'v1.csv', and 'v2.csv' based on the required ticket attributes. It is essential that I do not overlook any useful information, and the retrieved parameters should remain unfolded.
    
    Action: CSVQA

    Action Input:  {example_matches}

    Observation: {example_data_description}

    Thought: I now have all the required information from the CSV, including avg_price, avg_pax, value_list, ratio_list, value_0_list, ratio_0_list, flight_capacity, capacity_consum, option_num and so on for the specified flights. I will now formulate the SBLP model in markdown format, following the example provided, including all constraints, retrieved information, and generated code. Remember that all the retrieved parameters should remain unfolded in the final answer.

    Final Answer:
### Objective Function

\[
\max \quad \sum_(l,k,j) avg\_price[(l,k,j)] \cdot x[(l,k,j)]
\]

### Constraints

#### 1. Capacity Constraints

For each flight (OD pair \( l \), departure time \( k \)):

\[
capacity\_consum \cdot x[(l,k,f)] + x[(l,k,l)] \leq flight\_capacity
\]

#### 2. Balance Constraints

For each OD pair \( l \):

\[
ratio\_0\_list[l] \cdot x_o[l] + \sum_k \sum_j ratio\_list[(l,k,j)] \cdot x[(l,k,j)] = avg\_pax[l]
\]

#### 3. Scale Constraints

For each ticket option \((l,k,j)\):

\[
x[(l,k,j)]/value\_list[(l,k,j)] - x_o[l]/value\_0\_list[l] \leq 0
\]

#### 4. Big M Constraints

For each ticket option \((l,k,j)\):

\[
x[(l,k,j)] \leq 10000000 \cdot y[(l,k)]
\]

#### 5. Cardinality Constraint

\[
\sum_(l,k) y[(l,k)] \leq option\_num
\]


#### 6. Flow Conservation Constraints

\[
\sum_(l,k)  \in sigma_inflow_A y[(l,k)] = \sum_(l,k)  \in sigma_outflow_A y[(l,k)]
\]
\[
\sum_(l,k)  \in sigma_inflow_B y[(l,k)] = \sum_(l,k)  \in sigma_outflow_B y[(l,k)]
\]
\[
\sum_(l,k)  \in sigma_inflow_C y[(l,k)] = \sum_(l,k)  \in sigma_outflow_C y[(l,k)]
\]

#### 7. Nonnegativity Constraints

\[
x[(l,k,j)] \geq 0, \quad x_o[l] \geq 0
\]
- x = avg_price.keys()
- x_o = avg_pax.keys()

#### 8. Binary Constraints

\[
y[(l,k)] is binary
\]

### Retrieved Information
{example_data_description}

### Generated Code

```python
import gurobipy as gp
from gurobipy import GRB
{example_data_description}
total_set = sigma_inflow_A + sigma_outflow_A + sigma_inflow_B + sigma_outflow_B + sigma_inflow_C + sigma_outflow_C
total_set = list(set(total_set)) 
model = gp.Model("sales_based_lp")
y = model.addVars(total_set, vtype=GRB.BINARY, name="y")  
x = model.addVars(avg_price.keys(), lb=0, name="x") 
x_o = model.addVars(value_0_list.keys(), lb=0, name="x_o")  

model.setObjective(gp.quicksum(avg_price[key] * x[key] for key in avg_price.keys()), GRB.MAXIMIZE)

paired_keys = []
for i in range(0, len(avg_price.keys()), 2):  
    if i + 1 < len(avg_price.keys()):  
        names = list(avg_price.keys())
        model.addConstr(
            capacity_consum* x[names[i]] + x[names[i + 1]] <= 187,
            name=f"capacity_constraint"
        )

for l in ratio_0_list.keys():
    temp = 0
    for key in ratio_list.keys():
        if l in key:
            temp += ratio_list[key] * x[key]
    model.addConstr(
        float(ratio_0_list[l]) * x_o[l] + temp == float(avg_pax[l])
    )

for i in value_list.keys():
    for l in ratio_0_list.keys():
        if l in i:
            model.addConstr(
                value_0_list[l]  * x[i] <= value_list[i] * x_o[l]
            )


for key in avg_price.keys():
    for lk in total_set:
       if lk.split(',')[0].strip('(').strip(')')==key.split(',')[0].strip('(').strip(')') and lk.split(',')[1].strip('(').strip(')')==key.split(',')[1].strip('(').strip(')'):
            model.addConstr(
                x[key] <= 10000000 * y[lk],  
                name=f"big_m_constraint"
            )

model.addConstr(gp.quicksum(y[lk] for lk in total_set) <= option_num,
        name=f"cardinality_constraint"
    )


model.addConstr(
    gp.quicksum(y[inflow] for inflow in sigma_inflow_A) == gp.quicksum(y[outflow] for outflow in sigma_outflow_A),
    name=f"flow_conservation_A"
)


model.addConstr(
    gp.quicksum(y[inflow] for inflow in sigma_inflow_B) == gp.quicksum(y[outflow] for outflow in sigma_outflow_B),
    name=f"flow_conservation_B"
)
    
model.addConstr(
    gp.quicksum(y[inflow] for inflow in sigma_inflow_C) == gp.quicksum(y[outflow] for outflow in sigma_outflow_C),
    name=f"flow_conservation_C"
)

model.optimize()


if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    for v in model.getVars():
        print(v.varName, v.x)
    print("Optimal objective value:", model.objVal)
else:
    print("No optimal solution found.")
    '''

    tools = [Tool(name="CSVQA", func=csv_qa_tool_flow, description="Retrieve flight data."),Tool(name="CName", func=retrieve_key_information, description="Retrieve flight information.")]

    llm = lx.build_llm(CFG, "data_agent")
    prefix = f"""You are an assistant that generates a SBLP mathematical model and the corresponding Gurobi Python code based on the user's description and provided CSV data. Your core task is to:
1. **STRICTLY AND COMPLETELY** follow the **EXACT FORMAT** of the **Few-Shot Example** provided below, including all **Markdown headings**, **LaTeX equations**, and the **Python code block**.
2. Use the `CName` and `CSVQA` tools to retrieve all necessary data.

**CRITICAL REQUIREMENT:** All retrieved parameters (e.g., avg_price, value_list, avg_pax) **MUST REMAIN UNFOLDED** in their dictionary form within the "Retrieved Information" section and in the Gurobi code.

Please refer to the following example and generate the answer in the **IDENTICAL FORMAT**:

{fewshot_example}

Note: You must retrieve all necessary information from the CSV file to generate the answer. When generating the final answer, output all required parameters (vectors and matrices) as a whole block of text.
"""

    suffix = """

    Begin!

    User Description: {input}
    {agent_scratchpad}"""


    agent2 = initialize_agent(
        tools,
        llm=llm,
        agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        **lx.agent_kwargs(CFG, prefix, suffix),
    )

    return agent2

def policy_sblp_flow_model_code(query):
    agent2 = FlowAgent(query)
    llm_code = lx.build_llm(CFG, "coder")
    result = agent2.invoke({"input": query})
    output_model = result['output']

    return output_model

def ProcessPolicyFlow(query):
    output_model = policy_sblp_flow_model_code(query)

    return output_model

## CA

In [ ]:
# ── LARGE-SCALE REWRITE of csv_qa_tool_CA ───────────────────────────────────
# Four things the small-scale version cannot do here:
#
#   1. OD codes.  It read them by character position --
#          origin = match[0][2];  destination = match[0][7]
#      which works only while every code is a single character. Against
#      ('865', '678') that yields ('8', ' ') and every downstream lookup
#      silently returns nothing. Now the regex captures the two codes itself.
#   2. Fare families.  It built exactly two keys per flight, `code_f` and
#      `code_l`. There are four families here, so the keys are generated from
#      LoadProducts().
#   3. Capacity consumption.  It searched for "Eco_flex ticket consumes" and
#      kept a single number. Our queries say "Eco_flexi", so the search missed
#      and it fell back to a default of 1.2 -- and it has no slot for the other
#      three coefficients anyway. Now all of them are parsed, and a query that
#      does not name every product raises.
#   4. Flight capacity.  It emitted the constant 187. Capacity is per flight
#      here: 'Y Seats/Week' from flight_capacity.csv, times the percentage the
#      query states. So `flight_capacity` becomes a dict keyed like the cells,
#      not a scalar.
#
# Key shapes are unchanged, so the model template still recognises them:
#     cell   "(<origin><destination>,<time>,<short>)"   short in f/s/v/l
#     OD     "<origin><destination>"
#     flight "(<origin><destination>,<time>)"

FLIGHT_RE = re.compile(
    r"\(OD\s*=\s*\(\s*'([^']+)'\s*,\s*'([^']+)'\s*\)\s*AND\s+"
    r"Departure\s*Time\s*=\s*'(\d{1,2}:\d{2})'\)")
CONSUME_RE = re.compile(r"each\s+(Eco_\w+)\s+ticket\s+consumes\s+([\d.]+)\s+unit")
HAIRCUT_RE = re.compile(r"capacity is (\d+(?:\.\d+)?)% of its weekly economy capacity")


def _short_code(product):
    """Eco_flexi -> f, Eco_standard -> s, Eco_value -> v, Eco_lite -> l."""
    return product.split('_')[-1][0]


# The instance under test. The agent composes its own Action Input rather than
# forwarding CName's output verbatim, and what it composes is not reliably the
# whole sentence: across three repeats it twice handed CSVQA a flight list with
# the capacity clause dropped, or with one of the four fare families missing.
#
# Which flights to look up is genuinely part of the tool call -- the agent
# chooses those. The capacity consumption and the haircut are not: they are
# stated once in the question and never vary within it. Making the tool depend
# on the model retyping them is a property of this harness, not of the method,
# and it fails in a way that reads like a modelling error.
CURRENT_QUERY = ""


def csv_qa_tool_CA(query: str):
    products = LoadProducts()
    v1, v2, demand, flight_df, capacity_df = LoadFiles()

    flights = FLIGHT_RE.findall(query)
    if not flights:
        raise ValueError("no (OD = (...) AND Departure Time='..') clause in the query")

    # Capacity consumption, one coefficient per fare family, straight from the
    # query text. A missing product is an error rather than a default: a wrong
    # capacity coefficient still solves, it just solves a different problem.
    # Flights from what the agent asked for; capacity terms from the question.
    stated = dict(CONSUME_RE.findall(query))
    if len(stated) < len(products) and CURRENT_QUERY:
        stated = dict(CONSUME_RE.findall(CURRENT_QUERY)) or stated
    capacity_consum = {}
    for product in products:
        hit = next((k for k in stated if k == product), None)
        if hit is None:
            hit = next((k for k in stated
                        if product.startswith(k) or k.startswith(product)), None)
        if hit is None:
            raise ValueError(f"the query states no capacity consumption for {product!r}; "
                             f"it names {sorted(stated)}")
        capacity_consum[_short_code(product)] = float(stated[hit])

    haircut = HAIRCUT_RE.search(query) or (
        HAIRCUT_RE.search(CURRENT_QUERY) if CURRENT_QUERY else None)
    capacity_fraction = float(haircut.group(1)) / 100.0 if haircut else 1.0

    cap_lookup = capacity_df.set_index(['Oneway_OD', 'Departure Time'])['Y Seats/Week']
    price_lookup = flight_df.set_index(
        ['Oneway_OD', 'Departure Time', 'Oneway_Product'])['Avg Price']

    avg_price, value_list, ratio_list = {}, {}, {}
    value_0_list, ratio_0_list, avg_pax, flight_capacity = {}, {}, {}, {}
    x, x_o = {}, {}

    for origin, destination, time in flights:
        od = str((origin, destination))
        code_o = f"{origin}{destination}"
        flight_key = f"({code_o},{time})"

        values, ratios, value_0, ratio_0 = generate_coefficients(od, time)
        for product in products:
            code = f"({code_o},{time},{_short_code(product)})"
            x[code] = f"x_{code_o}_{time}_{_short_code(product)}"
            # float, not the raw string the retriever's page_content yields.
            # The exemplar's objective does `avg_price[key] * x[key]`, which
            # cannot multiply a str by a Gurobi Var; the original only wrapped
            # float() around the balance-constraint terms, so the two halves
            # of the template disagreed about the type.
            retrieved = float(retrieve_avg_price(od, time, product))
            # Guard the one step the offline verifier cannot cover. The metadata
            # filter narrows to this exact (OD, time), and the vector search then
            # picks among four page_contents that differ only by the product
            # name -- Eco_standard and Eco_value are far closer in embedding
            # space than the small-scale set's Eco_flexi / Eco_lite ever were.
            # A wrong pick would put a real price on the wrong fare family and
            # the LP would still solve. One dataframe lookup makes that loud.
            expected = float(price_lookup[(od, time, product)])
            if abs(retrieved - expected) > 1e-9:
                raise AssertionError(
                    f"retrieval returned {retrieved} for {od} {time} {product}, "
                    f"but flight.csv says {expected} -- the vector search picked "
                    f"the wrong fare family")
            avg_price[code] = retrieved
            value_list[code] = values[product]
            ratio_list[code] = ratios[product]

        value_0_list[code_o] = value_0
        ratio_0_list[code_o] = ratio_0
        x_o[code_o] = f"x_{code_o}_o"
        flight_capacity[flight_key] = round(
            float(cap_lookup[(od, time)]) * capacity_fraction, 2)

    for origin, destination, _ in flights:
        od = str((origin, destination))
        avg_pax[f"{origin}{destination}"] = float(retrieve_avg_pax(od))

    # One assignment per line, no leading whitespace, no space around "=".
    # This block is pasted straight into the python code fence of the
    # few-shot example, so it has to be valid Python: the original emitted
    # " value_list =..." with a leading space, which is an IndentationError the
    # moment anyone runs the exemplar as written.
    doc = "\n".join([
        f"avg_price={avg_price}",
        f"value_list={value_list}",
        f"ratio_list={ratio_list}",
        f"value_0_list={value_0_list}",
        f"ratio_0_list={ratio_0_list}",
        f"avg_pax={avg_pax}",
        f"capacity_consum={capacity_consum}",
        f"flight_capacity={flight_capacity}",
    ]) + "\n"
    return doc


def CA_Agent(query):

    # LARGE-SCALE: the original exemplar is written over the toy ODs ('A','B'),
    # which do not exist in this v1.csv -- building the few-shot from it now
    # raises rather than silently returning zeros. This one names two real
    # flights and is worded identically to the queries under test.
    loader = CSVLoader(file_path="Large_Scale_Or_Files/RAG_Example_SBLP_CA_LargeScale.csv", encoding="utf-8")
    data = loader.load()
    documents = data
    embeddings = EMBEDDINGS
    vectors = FAISS.from_documents(documents, embeddings)
    retriever = lx.build_retriever(CFG, vectors, "air_examples")
    similar_results = retrieve_similar_docs(query,retriever)
    problem_description = similar_results[0]['content'].replace("prompt:", "").strip()  
    example_matches = retrieve_key_information(problem_description)
    example_data_description = csv_qa_tool_CA(example_matches)
    example_data_description = example_data_description.replace('{', '{{')
    example_data_description = example_data_description.replace('}', '}}')    
    fewshot_example = f'''
    Question: {problem_description}

    Thought: I need to retrive the required OD and Departure Time information.

    Action: CName

    Action Input: {problem_description}

    Observation: {example_matches}

    Thought: I need to extract all relevant information from the files 'flight.csv', 'od_demand.csv', 'v1.csv', and 'v2.csv' based on the required ticket attributes. It is essential that I do not overlook any useful information, and the retrieved parameters should remain unfolded.
    
    Action: CSVQA

    Action Input:  {example_matches}

    Observation: {example_data_description}

    Thought: I now have all the required information from the CSV, including avg_price, avg_pax, value_list, ratio_list, value_0_list, ratio_0_list, flight_capacity, capacity_consum, option_num and so on for the specified flights. I will now formulate the SBLP model in markdown format, following the example provided, including all constraints, retrieved information, and generated code. Remember that all the retrieved parameters should remain unfolded in the final answer.


    Final Answer:
### Objective Function

\[
\max \quad \sum_(l,k,j) avg\_price[(l,k,j)] \cdot x[(l,k,j)]
\]

### Constraints

#### 1. Capacity Constraints

For each flight (OD pair \( l \), departure time \( k \)), summed over every
fare family \( j \). Both the per-family consumption and the right-hand side
are looked up, not constants:

\[
\sum_j capacity\_consum[j] \cdot x[(l,k,j)] \leq flight\_capacity[(l,k)]
\]

#### 2. Balance Constraints

For each OD pair \( l \):

\[
ratio\_0\_list[l] \cdot x_o[l] + \sum_k \sum_j ratio\_list[(l,k,j)] \cdot x[(l,k,j)] = avg\_pax[l]
\]

#### 3. Scale Constraints

For each ticket option \((l,k,j)\):

\[
x[(l,k,j)]/value\_list[(l,k,j)] - x_o[l]/value\_0\_list[l] \leq 0
\]

#### 4. Nonnegativity Constraints

\[
x[(l,k,j)] \geq 0, \quad x_o[l] \geq 0
\]
- x = avg_price.keys()
- x_o = avg_pax.keys()

### Generated Code

```python
import gurobipy as gp
from gurobipy import GRB
{example_data_description}
model = gp.Model("sales_based_lp")
x = model.addVars(avg_price.keys(), lb=0, name="x") 
x_o = model.addVars(value_0_list.keys(), lb=0, name="x_o")  

model.setObjective(gp.quicksum(avg_price[key] * x[key] for key in avg_price.keys()), GRB.MAXIMIZE)

# dict() and string concatenation are used here instead of a brace literal and
# an f-string, on purpose. This text passes through TWO brace-eating layers:
# first the f-string that builds fewshot_example, then langchain's
# PromptTemplate, which treats any surviving brace-name-brace as a prompt
# variable and refuses to start the agent. Quadruple-escaping would work and
# would be unreadable. The original template contains no literal braces at all,
# and neither does this one -- including this comment, which sits inside the
# f-string and would be evaluated like any other part of it.
#
# Keys look like "(<origin><destination>,<time>,<short>)", so everything before
# the last comma identifies the flight and the last field is the fare family.
# Group by flight, then sum that flight's families with their own consumption
# coefficients against that flight's own capacity.
flights = dict()
for key in avg_price.keys():
    head, short = key.rsplit(',', 1)
    flights.setdefault(head + ')', []).append(key)

for flight_key, keys in flights.items():
    model.addConstr(
        gp.quicksum(capacity_consum[k.rsplit(',', 1)[1].rstrip(')')] * x[k] for k in keys)
        <= flight_capacity[flight_key],
        name="capacity_constraint_" + flight_key
    )

for l in ratio_0_list.keys():
    temp = 0
    for key in ratio_list.keys():
        if l in key:
            temp += ratio_list[key] * x[key]
    model.addConstr(
        float(ratio_0_list[l]) * x_o[l] + temp == float(avg_pax[l])
    )

for i in value_list.keys():
    for l in ratio_0_list.keys():
        if l in i:
            model.addConstr(
                value_0_list[l]  * x[i] <= value_list[i] * x_o[l]
            )

model.optimize()

if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    for v in model.getVars():
        print(v.varName, v.x)
    print("Optimal objective value:", model.objVal)
else:
    print("No optimal solution found.")
    '''

    tools = [Tool(name="CSVQA", func=csv_qa_tool_CA, description="Retrieve flight data."),Tool(name="CName", func=retrieve_key_information, description="Retrieve flight information.")]

    llm = lx.build_llm(CFG, "data_agent")
    prefix = f"""You are an assistant that generates a SBLP mathematical model and the corresponding Gurobi Python code based on the user's description and provided CSV data.  When you need to retrieve information from the CSV file, please use the CSVQA tool. When you need to retrieve flight information, please use the CName tool.

    Please refer to the following example and generate the answer in the same format:

    {fewshot_example}

    Note: Please retrieve all neccessary information from the CSV file to generate the answer.

    The Generated Code section must be ONE python code block that runs on its own,
    with nothing before it and nothing after it that it depends on. Inside that
    block, every dictionary -- avg_price, value_list, ratio_list, value_0_list,
    ratio_0_list, avg_pax, capacity_consum, flight_capacity -- must be written out
    in full, with every key and every value exactly as the CSVQA tool returned them.

    Do not abbreviate. Never replace a dictionary with an ellipsis placeholder, a
    "see above" comment, a "same as before" comment, or a "truncated" comment. Do
    not split the parameters into a separate code block from the model. Do not
    repeat the parameters anywhere outside that one block. Length is not a reason
    to shorten them: a block whose dictionaries are abbreviated cannot be executed
    and counts as no answer at all.
    """

    suffix = """

        Begin!

        User Description: {input}
        {agent_scratchpad}"""


    agent2 = initialize_agent(
    tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    **lx.agent_kwargs(CFG, prefix, suffix),
    )

    return agent2

def conversational_chat(query):
    agent2 = CA_Agent(query)
    result = agent2.invoke({"input": query})
    output = result['output']
    return output

def get_answer(query):
    agent2 = CA_Agent(query)
    result = agent2.invoke({"input": query})
    output_model = result['output']
    return output_model
def ProcessCA(query):
    CA_model = get_answer(query)
    return CA_model


# Combined Function

In [ ]:
def Process_Input(query):
  Type, output_model = None, None
  with lx.stage("classification"):
      category_original = Problemtype(query)
  print(f"Problem type classification finished, it belongs to {category_original}.")
  if "Sales-Based Linear Programming" in    category_original:
    print("Processing AirNRM queries")
    if "flow conservation constraints" in query or "flow conservation constraint" in query:
        print('----------Flow Constraints----------')
        print("Recommend Optimal Flights With Flow Conervation Constraints")
        with lx.stage("modeling"):
            output_model = ProcessPolicyFlow(query)
        Type = "Policy_Flow"

    else:
        print('----------CA----------')
        print("Only Develop Mathematic Formulations. No Recommendation for Flights.")
        with lx.stage("modeling"):
            output_model = ProcessCA(query)
        Type = "CA"
  if output_model is None:
      raise ValueError(
          f"unroutable problem type: {category_original!r}")
  return Type, output_model

# Run the test

In [ ]:


import re as _re
from tqdm import tqdm


_FENCE_RE = _re.compile(r"```([A-Za-z0-9_+-]*)\n(.*?)```", _re.DOTALL)
# "avg_price = {...}  # (see above)" -- a set holding an Ellipsis, never real data.
_PLACEHOLDER_RE = _re.compile(r"^\s*\w+\s*=\s*\{\s*\.\.\.\s*\}")


def _find_dict_assignment(text, name):
    """Return the full `name = {...}` statement from anywhere in the answer.

    Brace-counted rather than regex-matched, because the dictionaries run to
    thousands of characters and may wrap. Placeholders are skipped, so this
    keeps looking until it finds a definition with real content.
    """
    for m in _re.finditer(rf"^{_re.escape(name)}\s*=\s*\{{", str(text), _re.M):
        depth = 0
        for i in range(m.end() - 1, len(text)):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    statement = text[m.start(): i + 1]
                    if "..." not in statement:
                        return statement
                    break
    return None


def _extract_gurobipy_code(text):
    """Reassemble the gurobipy program from a model answer.

    The original returned the FIRST fenced block mentioning gurobi. That works
    while the answer is one block, which is what happens on the small instances.
    On the large ones gpt-4.1 splits the answer up: instance 14 came back as
    nine python fences -- one per parameter dict, then the model code -- and in
    that last fence the parameters were abbreviated to

        avg_price = {...}  # (see above)

    Returning only that fence loses every number and the code dies with
    "'set' object has no attribute 'keys'", because {...} is a set containing
    an Ellipsis. Scored as a modelling failure; it was not one. Reassembled by
    hand, that same answer solves to 7,558,440.98 against a label of
    7,558,440.98 -- 421 variables, 519 constraints, exact.

    So: concatenate the python fences in order, up to and including the last
    one that builds a model, and drop the placeholder redefinitions. Nothing is
    invented -- every number comes from a fence the model wrote.
    """
    if not text:
        return None
    blocks = [(lang.lower(), body) for lang, body in _FENCE_RE.findall(str(text))]
    python = [body for lang, body in blocks if lang in ("", "python", "py")]
    if not python:
        return None

    # Trailing fences are often sample output rather than program text.
    last = max((i for i, b in enumerate(python) if "gurobi" in b.lower()),
               default=len(python) - 1)

    lines = []
    placeholders = []
    for body in python[: last + 1]:
        for line in body.splitlines():
            hit = _PLACEHOLDER_RE.match(line)
            if hit:
                placeholders.append(line.split("=")[0].strip())
                continue
            lines.append(line)
    code = "\n".join(lines)

    # A placeholder whose real definition is nowhere in the fences is not lost:
    # the exemplar asks for the parameters twice -- once under "### Retrieved
    # Information" as plain markdown, once inside the code block -- and at 80
    # flights that is 26k tokens of the same dictionaries. The model reasonably
    # writes them out once, in the markdown, and abbreviates the second copy.
    # So look for the definition in the answer as a whole before giving up.
    recovered = []
    for name in placeholders:
        if _re.search(rf"^{_re.escape(name)}\s*=\s*\{{\s*['\"]", code, _re.M):
            continue                       # a real definition survived a fence
        statement = _find_dict_assignment(str(text), name)
        if statement:
            code = statement + "\n" + code
            recovered.append(name)

    code = code.strip()

    # Say so, loudly and greppably. The prompt now demands one self-contained
    # block; recovery firing at all means the model did not comply, and that is
    # a fact about the run, not a detail of the parser. Silently repairing it
    # would hide exactly the behaviour worth measuring.
    if placeholders:
        print(f"  [extract] FRAGMENTED-OUTPUT  blocks={last + 1} "
              f"placeholders={len(placeholders)} recovered={len(recovered)}"
              + ("" if len(recovered) == len(placeholders)
                 else "  <-- NOT fully recoverable"))

    # Every parameter the model template uses has to be bound to a real dict.
    # Without this the failure surfaces later as an opaque NameError or
    # "'set' object has no attribute 'keys'", which reads like a modelling
    # mistake rather than a truncated answer.
    required = ["avg_price", "value_list", "ratio_list", "value_0_list",
                "ratio_0_list", "avg_pax", "capacity_consum", "flight_capacity"]
    missing = [n for n in required
               if not _re.search(rf"^{n}\s*=\s*\{{\s*['\"]", code, _re.M)]
    if missing:
        print(f"  [extract] INCOMPLETE-CODE  no usable definition for "
              f"{', '.join(missing)} anywhere in the answer")

    return code or None


def Batch_Process_Queries(df, query_column='Query', log=LOG, cfg=CFG,
                          gold_column='Label-objective'):
    """
    Instrumented batch loop for the Air-NRM notebooks.

    Every query is wrapped in log.instance(...), so tokens / LLM calls /
    tool calls / cost / latency are recorded per instance and land in
    runs/<run_id>/instances.jsonl. Failures still produce a row, so the
    output frame always has the same length as the input frame.

    Two things beyond the original loop, both so the log is self-sufficient:
      * the ground-truth objective is carried through when the query file has
        one (see link_air_nrm_labels.py), otherwise score_runs.py has nothing
        to compare against;
      * the gurobipy block is extracted per instance rather than only in the
        run-everything cell downstream.
    """
    results = []
    has_gold = gold_column in df.columns

    for pos, (idx, row) in enumerate(tqdm(list(df.iterrows()),
                                          desc="Processing Queries")):
        query = row[query_column]
        # Label the instance by its position in the source file, not by
        # position in this slice -- ids have to stay comparable across batches.
        instance_id = int(idx) if isinstance(idx, (int,)) else pos
        gold = row[gold_column] if has_gold else None
        with log.instance(instance_id=instance_id, query=query,
                          gold_objective=gold) as rec:
            category, output_model = None, None
            try:
                # Record the instance so csv_qa_tool_CA can fall back to it for the
                # capacity terms when the agent's Action Input has dropped them.
                globals()['CURRENT_QUERY'] = query
                category, output_model = Process_Input(query)
                rec.set(pred_type=category, model_output=output_model,
                        code_output=_extract_gurobipy_code(output_model))
            except Exception as e:
                print(f"[{instance_id}] failed: {type(e).__name__}: {e}")
                rec.fail(e)                      # status -> "failed" in the log
        results.append({
            "Category": category,
            "Original_Query": query,
            "Output": output_model,
        })

    return pd.DataFrame(results)


In [ ]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import re
import numpy as np

def extract_objective(log_text):
   
    if "Optimal objective" in log_text:
        pattern = r"Optimal objective value: ([-+]?\d*\.?\d+)"
        match = re.search(pattern, log_text)
        if match:
            return float(match.group(1))

    elif "Best objective" in log_text:
        pattern = r"Best objective\s+([-+]?\d*\.?\d+e[-+]?\d+)"
        match = re.search(pattern, log_text, re.IGNORECASE)
        if match:
            return float(match.group(1))
    return None  



def gain_obj(df_out):
    output_code = df_out['Code'].tolist()
    obj = []
    i = 1

    for code in output_code:
        print(i)
        namespace = {'gp': gp, 'GRB': GRB}
        
  
        log_output = []
        

        class LogCapture:
            def write(self, message):
                log_output.append(message)

            def flush(self):  
                pass
            
        import sys
        log_capture = LogCapture()
        original_stdout = sys.stdout  
        sys.stdout = log_capture  

        try:
            exec(code, namespace)
        except Exception as e:
            log_output.append(f"Error: {e}\n")
            print(f"Error executing code block {i}: {e}")
        finally:
            sys.stdout = original_stdout 

    
        log_text = ''.join(log_output)

    
        optimal_value = extract_objective(log_text)
        if optimal_value is not None:
            obj.append(optimal_value)
            print(f"Optimal Value: {optimal_value}")
        else:

            obj.append("No optimal value found.")
            print("No optimal value found.")
        i += 1
    return obj


### Run CA

In [ ]:
# LARGE-SCALE: 15 instances, 112-421 variables each.
# run_notebook.py overrides this via --data / --rows; the path here is
# what a plain 'Run All' would use.
df = pd.read_csv('Test_Dataset/Air_NRM/large_scale/query_CA.csv')
result_df = Batch_Process_Queries(df)
result_df.to_csv(LOG.dir / "example_answer_CA_largescale_GPT4.1.csv", index=False)

Code = []
for i in range(len(result_df['Output'])):
    text = result_df['Output'][i]
    pattern = r'```python(.*?)```'
    matches = re.findall(pattern, text, re.DOTALL)
    code = matches[0]
    if 'gurobipy' in code:
        Code.append(code)
    else:
        if len(matches) > 0:
            Code.append(matches[1])
        else:
            Code.append("No code found")

code_df = pd.DataFrame(Code, columns=['Code'])
code_df.to_csv(LOG.dir / "example_code_CA_test_GPT4.1_New.csv", index=False)

obj = gain_obj(code_df)
obj_df = pd.DataFrame({'Optimal Value': obj})
obj_df.to_csv(LOG.dir / "OBJ_CA_4.1.csv", index=False)

combined_df = pd.concat([result_df, code_df,obj_df], axis=1)
combined_df.to_csv(LOG.dir / "final_answer_CA_test_GPT4.1_New.csv", index=False)

### Run NP Flow

In [ ]:
# LARGE-SCALE: there is no NP-Flow query set for this dataset -- only
# query_CA.csv exists under large_scale/. Left commented rather than
# deleted so the cell layout still lines up with the small-scale notebook.
# df = pd.read_csv('Test_Dataset/Air_NRM/small_scale/query_NP_Flow.csv')
# result_df = Batch_Process_Queries(df)
# result_df.to_csv(LOG.dir / "example_answer_NP_Flow_test_GPT4.1_New_V2.csv", index=False)

# Code = []
# for i in range(len(result_df['Output'])):
#     text = result_df['Output'][i]
#     pattern = r'```python(.*?)```'
#     matches = re.findall(pattern, text, re.DOTALL)
#     code = matches[0]
#     if 'gurobipy' in code:
#         Code.append(code)
#     else:
#         if len(matches) > 0:
#             Code.append(matches[1])
#         else:
#             Code.append("No code found")

# code_df = pd.DataFrame(Code, columns=['Code'])
# code_df.to_csv(LOG.dir / "example_code_NP_Flow_test_GPT4.1_New_V2.csv", index=False)

# obj = gain_obj(code_df)
# obj_df = pd.DataFrame({'Optimal Value': obj})
# obj_df.to_csv(LOG.dir / "OBJ_NP_New_4.1_V2.csv", index=False)

# combined_df = pd.concat([result_df, code_df,obj_df], axis=1)
# combined_df.to_csv(LOG.dir / "final_answer_NP_Flow_test_GPT4.1_New_V2.csv", index=False)

In [ ]:
# df = pd.read_csv('Test_Dataset/Air_NRM/small_scale/query_NP_Flow.csv')
# result_df = Batch_Process_Queries(df)
# result_df.to_csv("example_answer_NP_Flow_test_GPT4.1_New_V1.csv", index=False)

# Code = []
# for i in range(len(result_df['Output'])):
#     text = result_df['Output'][i]
#     pattern = r'```python(.*?)```'
#     matches = re.findall(pattern, text, re.DOTALL)
#     code = matches[0]
#     if 'gurobipy' in code:
#         Code.append(code)
#     else:
#         if len(matches) > 0:
#             Code.append(matches[1])
#         else:
#             Code.append("No code found")

# code_df = pd.DataFrame(Code, columns=['Code'])
# code_df.to_csv("example_code_NP_Flow_test_GPT4.1_New_V1.csv", index=False)

# obj = gain_obj(code_df)
# obj_df = pd.DataFrame({'Optimal Value': obj})
# obj_df.to_csv('OBJ_NP_New_4.1.csv_V1', index=False)

# combined_df = pd.concat([result_df, code_df,obj_df], axis=1)
# combined_df.to_csv("final_answer_NP_Flow_test_GPT4.1_New_V1.csv", index=False)

In [ ]:
# df = pd.read_csv('Test_Dataset/Air_NRM/small_scale/query_NP_Flow.csv')
# result_df = Batch_Process_Queries(df)
# result_df.to_csv("example_answer_NP_Flow_test_GPT4.1_New.csv", index=False)

# Code = []
# for i in range(len(result_df['Output'])):
#     text = result_df['Output'][i]
#     pattern = r'```python(.*?)```'
#     matches = re.findall(pattern, text, re.DOTALL)
#     code = matches[0]
#     if 'gurobipy' in code:
#         Code.append(code)
#     else:
#         if len(matches) > 0:
#             Code.append(matches[1])
#         else:
#             Code.append("No code found")

# code_df = pd.DataFrame(Code, columns=['Code'])
# code_df.to_csv("example_code_NP_Flow_test_GPT4.1_New.csv", index=False)

# obj = gain_obj(code_df)
# obj_df = pd.DataFrame({'Optimal Value': obj})
# obj_df.to_csv('OBJ_NP_New_4.1.csv', index=False)

# combined_df = pd.concat([result_df, code_df,obj_df], axis=1)
# combined_df.to_csv("final_answer_NP_Flow_test_GPT4.1_New.csv", index=False)

In [ ]:
# ---------------------------------------------------------------
# Finish the run: writes runs/<run_id>/summary.json and prints the
# totals. Safe to run after every slice.
# ---------------------------------------------------------------
agg = LOG.close()

import json
rows = [json.loads(l) for l in open(LOG.dir_path / 'instances.jsonl')]
est = sum(r['summary'].get('n_estimated_token_calls', 0) for r in rows)
print(f"\ntoken source: {'provider (good)' if est == 0 else f'{est} calls ESTIMATED -- report as estimates'}")
print('run dir:', LOG.dir_path)
